# WaveForge — Real GPU Benchmark & Auto-Publish

This notebook:
1. Runs WaveForge on the **real T4 GPU** — measures actual throughput
2. Loads pre-measured CPU+Meep results from the repo
3. Generates a comparison chart with **both real GPU and CPU numbers**
4. Commits the chart back to GitHub automatically

> **Setup:** Runtime → Change runtime type → **T4 GPU** → Run all cells

In [ ]:
# Step 1: Check GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')


In [ ]:
# Step 2: Clone repo and install
!git clone https://github.com/shahzaibshazoo/cuda-meep.git
!pip install torch numpy matplotlib --quiet


In [ ]:
# Step 3: Setup
import sys, time, json
import numpy as np
sys.path.insert(0, '/content/cuda-meep/src')
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only'
print(f'PyTorch {torch.__version__}')
print(f'Device: {DEVICE} — {GPU_NAME}')
if DEVICE == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')


In [ ]:
# Step 4: Run WaveForge GPU benchmark
from core import YeeGrid, FieldSet, MurABC, GaussianPulse, PointSource, SourceCollection, FDTD2D

GRID_SIZES = [64, 128, 256, 512]
N_WARMUP   = 30
N_STEPS    = 300
DX         = 1e-3

def bench_waveforge(N, device):
    grid     = YeeGrid(N, N, dx=DX, dy=DX, device=device)
    fields   = FieldSet(grid)
    boundary = MurABC(grid, fields.Hz)
    pulse    = GaussianPulse(amplitude=1.0, sigma=30*grid.dt)
    src      = PointSource(pulse, N//2, N//2, 'Hz', grid=grid, N_steps=N_WARMUP+N_STEPS)
    sim      = FDTD2D(grid, fields, boundary, SourceCollection([src]), n_check=1000)
    sim.run(N_WARMUP)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    sim.run(N_STEPS)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return N_STEPS * N * N / elapsed / 1e6, elapsed / N_STEPS * 1000

gpu_results = []
print(f'--- WaveForge GPU ({GPU_NAME}) ---')
for N in GRID_SIZES:
    m, ms = bench_waveforge(N, DEVICE)
    gpu_results.append({'N': N, 'mcells_s': round(m, 2), 'ms_step': round(ms, 4)})
    print(f'  {N:4d}   {m:8.1f} Mcells/s   {ms:8.3f} ms/step')
print('GPU benchmark complete!')


In [ ]:
# Step 5: Load pre-measured CPU + Meep results
with open('/content/cuda-meep/benchmarks/cpu_results.json') as f:
    cpu_data = json.load(f)

cpu_key = 'waveforge_cpu' if 'waveforge_cpu' in cpu_data else 'cuda_meep_cpu'
waveforge_cpu = cpu_data[cpu_key]
meep_cpu      = cpu_data['meep_cpu']

print(f'CPU results from: {cpu_data["meta"]["date"][:10]}')
print('--- WaveForge CPU (pre-measured) ---')
for r in waveforge_cpu:
    print(f'  {r["N"]:4d}   {r["mcells_s"]:8.1f} Mcells/s')
print('--- Meep CPU (pre-measured) ---')
for r in meep_cpu:
    print(f'  {r["N"]:4d}   {r["mcells_s"]:8.1f} Mcells/s')


In [ ]:
# Step 6: Results table
gpu_mc  = [r['mcells_s'] for r in gpu_results]
cpu_mc  = [r['mcells_s'] for r in waveforge_cpu]
meep_mc = [r['mcells_s'] for r in meep_cpu]
speedups_meep = [g/m for g,m in zip(gpu_mc, meep_mc)]
speedups_cpu  = [g/c for g,c in zip(gpu_mc, cpu_mc)]

print('='*72)
print(f'  Grid    WaveForge GPU    WaveForge CPU    Meep CPU    GPU/Meep')
print(f'          ({GPU_NAME[:14]})')
print('='*72)
for i, N in enumerate(GRID_SIZES):
    s = speedups_meep[i]
    star = ' WINNER' if s >= 5 else ''
    print(f'  {N}x{N}    {gpu_mc[i]:>13.1f}    {cpu_mc[i]:>13.1f}    {meep_mc[i]:>8.1f}    {s:>6.1f}x{star}')
print('='*72)
avg = sum(speedups_meep[i] for i,N in enumerate(GRID_SIZES) if N >= 256) / 2
print(f'Average GPU speedup over Meep at 256+: {avg:.1f}x  (REAL MEASURED)')


In [ ]:
# Step 7: Generate comparison chart with real numbers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'WaveForge vs Meep — REAL Measured Benchmark ({GPU_NAME})',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(GRID_SIZES, gpu_mc,  'o-', color='#2ca02c', lw=2.5, ms=9,
        label=f'WaveForge GPU ({GPU_NAME})')
ax.plot(GRID_SIZES, cpu_mc,  's-', color='steelblue', lw=2, ms=8,
        label='WaveForge CPU (laptop, pre-measured)')
ax.plot(GRID_SIZES, meep_mc, '^-', color='#d62728', lw=2, ms=8,
        label='Meep CPU (laptop, pre-measured)')
peak_i = gpu_mc.index(max(gpu_mc))
ax.annotate(f'{gpu_mc[peak_i]:.0f} Mcells/s',
            (GRID_SIZES[peak_i], gpu_mc[peak_i]),
            textcoords='offset points', xytext=(5,10),
            fontsize=10, color='#2ca02c', fontweight='bold')
ax.set(xlabel='Grid size', ylabel='Throughput (Mcells/s)', title='Throughput Comparison')
ax.set_xticks(GRID_SIZES); ax.set_xticklabels([f'{N}x{N}' for N in GRID_SIZES])
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
x = range(len(GRID_SIZES)); w = 0.35
c_m = ['#d62728' if s<1 else '#2ca02c' for s in speedups_meep]
c_c = ['#d62728' if s<1 else 'steelblue' for s in speedups_cpu]
b1 = ax2.bar([i-w/2 for i in x], speedups_meep, w, color=c_m, alpha=0.85, label='vs Meep')
b2 = ax2.bar([i+w/2 for i in x], speedups_cpu,  w, color=c_c, alpha=0.85, label='vs WaveForge CPU')
ax2.bar_label(b1, [f'{s:.1f}x' for s in speedups_meep], fontsize=11, padding=3)
ax2.bar_label(b2, [f'{s:.1f}x' for s in speedups_cpu],  fontsize=11, padding=3)
ax2.axhline(1, color='black', ls='--', alpha=0.3)
ax2.set(ylabel='Speedup', title=f'GPU Speedup (REAL numbers)')
ax2.set_xticks(list(x)); ax2.set_xticklabels([f'{N}x{N}' for N in GRID_SIZES])
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
chart_path = '/content/cuda-meep/assets/benchmark_real_gpu.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved: {chart_path}')


In [ ]:
# Step 8: Save GPU results JSON
import datetime

results = {
    'meta': {
        'date':     datetime.datetime.now().isoformat(),
        'gpu':      GPU_NAME,
        'platform': 'Google Colab T4',
        'torch':    torch.__version__,
        'n_warmup': N_WARMUP,
        'n_steps':  N_STEPS,
        'measured': True
    },
    'waveforge_gpu':    gpu_results,
    'waveforge_cpu':    waveforge_cpu,
    'meep_cpu':         meep_cpu,
    'speedup_vs_meep':  {str(N): round(g/m, 2)
                         for N,g,m in zip(GRID_SIZES, gpu_mc, meep_mc)},
}

json_path = '/content/cuda-meep/benchmarks/gpu_results.json'
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to: {json_path}')
print('Speedups:', results['speedup_vs_meep'])


In [ ]:
# Step 9: Commit results back to GitHub
# You need a GitHub Personal Access Token with 'Contents: Read & Write' permission.
# Get one at: GitHub -> Settings -> Developer settings -> Personal access tokens -> Fine-grained
# Select repo: cuda-meep, permission: Contents: Read and write

GITHUB_TOKEN = ''  # <-- PASTE YOUR TOKEN HERE

import os
os.chdir('/content/cuda-meep')
!git config user.name 'Shahzaib Ur Rehman'
!git config user.email 'shahzaib@waveforge.io'

if GITHUB_TOKEN:
    import subprocess
    subprocess.run(['git', 'remote', 'set-url', 'origin',
                    f'https://{GITHUB_TOKEN}@github.com/shahzaibshazoo/cuda-meep.git'])
    !git add benchmarks/gpu_results.json assets/benchmark_real_gpu.png
    !git commit -m 'Add REAL T4 GPU benchmark results (measured on Google Colab)'
    !git push origin main
    print('Pushed to GitHub!')
    print('Chart URL: https://raw.githubusercontent.com/shahzaibshazoo/cuda-meep/main/assets/benchmark_real_gpu.png')
else:
    print('Set GITHUB_TOKEN above then re-run this cell.')
    print('Or download files manually from the Files panel (left sidebar).')
    print('Files to download:')
    print('  /content/cuda-meep/benchmarks/gpu_results.json')
    print('  /content/cuda-meep/assets/benchmark_real_gpu.png')


In [ ]:
# Step 10: Final summary
from IPython.display import Image, display
print('='*55)
print(f'WaveForge REAL GPU Results — {GPU_NAME}')
print('='*55)
for i, N in enumerate(GRID_SIZES):
    s = speedups_meep[i]
    bar = '#' * int(s * 2)
    print(f'  {N}x{N}  {gpu_mc[i]:>7.1f} Mcells/s  {s:>5.1f}x  {bar}')
print('='*55)
print(f'Peak: {max(gpu_mc):.0f} Mcells/s  |  Best speedup: {max(speedups_meep):.1f}x over Meep')
display(Image('/content/cuda-meep/assets/benchmark_real_gpu.png'))
